# From First Principles to Modern Large Language Models

**Author:** Paul Leyva

An unbroken chain of derivation from set theory to modern LLMs.


## Table of contents

### A — Foundations
- Chapter 1: Sets, functions, logic, proofs
- Chapter 2: Numbers, sequences, limits, completeness
- Chapter 3: Continuity, univariate differentiation, chain rule
- Chapter 4: Multivariate calculus: partials, gradients, Jacobians
- Chapter 5: Linear algebra I: vector spaces, basis, linear maps
- Chapter 6: Linear algebra II: inner products, norms, eigenvalues, SVD
- Chapter 7: Convexity and optimization; gradient descent convergence

### B — Probability and Information
- Chapter 8: Probability foundations: sample spaces, sigma-algebras, Kolmogorov axioms
- Chapter 9: Random variables, distributions, CDF/PMF/PDF
- Chapter 10: Expectation, variance, covariance; Jensen's inequality
- Chapter 11: Information theory: self-information, entropy, cross-entropy, KL
- Chapter 12: Statistical inference: likelihood, MLE, ERM, bias-variance

### C — Stochastic Optimization
- Chapter 13: SGD: stochastic-approximation theorem; mini-batching; convergence sketch
- Chapter 14: Momentum, RMSProp, AdamW: derivation and bias-correction proof

### D — Neural Networks
- Chapter 15: MLPs as compositional functions; universal approximation
- Chapter 16: Activation functions: ReLU/GELU/softmax with derivatives
- Chapter 17: Loss functions: MSE, cross-entropy; gradients from first principles
- Chapter 18: Backpropagation: chain rule applied; reverse-mode AD as a graph algorithm

### E — Sequence Models and Attention
- Chapter 19: Embeddings: token to vector; lookup as a linear map; weight tying
- Chapter 20: RNN intuition; vanishing-gradient proof; why we need attention
- Chapter 21: Scaled dot-product attention: derivation, softmax-temperature analysis
- Chapter 22: Multi-head attention: parallel heads as concat-then-project; complexity
- Chapter 23: Transformer block: residual + LayerNorm/RMSNorm + FFN + attention; gradient-flow argument
- Chapter 24: Positional encoding: sinusoidal derivation, RoPE construction

### F — Pre-training
- Chapter 25: Causal masking; next-token prediction loss as MLE on the empirical distribution
- Chapter 26: Tokenization: BPE algorithm; greedy merge correctness
- Chapter 27: Pre-training pipeline: AdamW + warmup + cosine decay + gradient clipping; tiny-GPT training run

### G — Post-training
- Chapter 28: SFT, RLHF (PPO/GRPO), and DPO; train + post-train a tiny GPT



# Block A — Foundations


# Chapter 1 — Sets, functions, logic, proofs

We need a precise grammar for membership, functions, and proof before we can define real numbers (Ch. 5), linear maps (Ch. 8), probability (Ch. 15), or token embeddings (Ch. 19).

**Key definitions.** A *set* $S$ is a collection of distinct elements; $x \in S$ means $x$ is an element of $S$. The *power set* is $\mathcal{P}(S) = \{T : T \subset S\}$. A *function* $f : A \to B$ assigns each $a \in A$ exactly one $f(a) \in B$.


In [ ]:
from itertools import chain, combinations

def power_set(S):
    S = list(S)
    return [set(c) for r in range(len(S) + 1) for c in combinations(S, r)]

S = {'a', 'b', 'c'}
P = power_set(S)
for T in P:
    print(sorted(T))
print('|P(S)| =', len(P), '   2**|S| =', 2 ** len(S))
assert len(P) == 2 ** len(S), 'power-set cardinality identity failed'


## De Morgan's laws

For $A, B \subset U$:
$$(A \cup B)^c = A^c \cap B^c, \qquad (A \cap B)^c = A^c \cup B^c.$$


In [ ]:
U = set(range(1, 9))
A = {1, 3, 5, 7}
B = {2, 3, 5, 7}

comp = lambda X: U - X

lhs1, rhs1 = comp(A | B), comp(A) & comp(B)
lhs2, rhs2 = comp(A & B), comp(A) | comp(B)

print('(A u B)^c =', sorted(lhs1), '  A^c n B^c =', sorted(rhs1))
print('(A n B)^c =', sorted(lhs2), '  A^c u B^c =', sorted(rhs2))
assert lhs1 == rhs1 and lhs2 == rhs2, 'De Morgan failed'
print('Both De Morgan identities verified.')


## Injection, surjection, bijection

$f : A \to B$ is *injective* iff $f(a_1) = f(a_2) \Rightarrow a_1 = a_2$, *surjective* iff every $b \in B$ has a preimage, *bijective* iff both.


In [ ]:
def is_injective(f, A):
    seen = {}
    for a in A:
        b = f[a]
        if b in seen:
            return False
        seen[b] = a
    return True

def is_surjective(f, A, B):
    return set(f[a] for a in A) == set(B)

def is_bijective(f, A, B):
    return is_injective(f, A) and is_surjective(f, A, B)

A = [0, 1, 2, 3]
B = [0, 1, 2, 3, 4]
f = {0: 1, 1: 3, 2: 0, 3: 4}   # injective into B, not surjective
print('f =', f)
print('injective?', is_injective(f, A))
print('surjective onto B?', is_surjective(f, A, B))
print('bijective A -> B?', is_bijective(f, A, B))

# A bijection A -> A
g = {0: 2, 1: 0, 2: 3, 3: 1}
print('\ng =', g)
print('bijective A -> A?', is_bijective(g, A, A))


## Induction

Claim: $\sum_{k=1}^{n} k = n(n+1)/2$.

*Base:* $n = 1$: LHS $= 1 =$ RHS. *Step:* assume $S(n) = n(n+1)/2$; then
$S(n+1) = S(n) + (n+1) = n(n+1)/2 + (n+1) = (n+1)(n+2)/2$.


In [ ]:
import numpy as np

def S_direct(n):
    return sum(range(1, n + 1))

def S_closed(n):
    return n * (n + 1) // 2

# Direct check up to n = 20
for n in range(1, 21):
    assert S_direct(n) == S_closed(n), f'mismatch at n={n}'
print('Direct verification 1..20: OK')

# Numerical induction-step check: S_closed(n+1) - S_closed(n) == n+1
ns = np.arange(1, 1001)
lhs = np.array([S_closed(n + 1) - S_closed(n) for n in ns])
rhs = ns + 1
assert np.array_equal(lhs, rhs), 'induction step failed'
print('Base case S(1) =', S_closed(1))
print('Induction step S(n+1) - S(n) = n+1 verified for n = 1..1000')


## Connection to LLMs

A vocabulary $\mathcal{V}$ is a finite set of tokens. The embedding map $E : \mathcal{V} \to \mathbb{R}^d$ is a function; its lookup-table implementation requires the index $\mathcal{V} \to \{0, \ldots, |\mathcal{V}|-1\}$ to be a **bijection**. We revisit this in Chapter 19.


In [ ]:
import numpy as np
np.random.seed(0)

vocab = ['<bos>', '<eos>', 'the', 'cat', 'sat', 'on', 'mat']
V = len(vocab)
d = 4

tok2id = {tok: i for i, tok in enumerate(vocab)}
id2tok = {i: tok for tok, i in tok2id.items()}

# Bijectivity of vocabulary index
assert len(set(tok2id.values())) == V                  # injective
assert set(tok2id.values()) == set(range(V))           # surjective onto {0,...,V-1}
print('vocabulary index is a bijection V <-> {0,...,V-1}')

E = np.random.randn(V, d).astype(np.float32)
print('embedding matrix shape:', E.shape)
for tok in ['cat', 'mat']:
    print(f'E[{tok!r}] =', E[tok2id[tok]])


# Chapter 2 — Numbers, sequences, limits, completeness

We build $\mathbb{N} \subset \mathbb{Z} \subset \mathbb{Q} \subset \mathbb{R}$ and isolate the **completeness axiom** of $\mathbb{R}$: every nonempty subset that is bounded above has a supremum in $\mathbb{R}$. This single axiom is what powers every convergence theorem we will need later for SGD and Adam.


## Convergence: $\varepsilon$–$N$ on $S_n = \sum_{k=1}^n 1/k^2 \to \pi^2/6$

We compute partial sums, the gap $|S_n - \pi^2/6|$, and for each $\varepsilon$ the smallest $N$ such that $n \geq N \Rightarrow |S_n - \pi^2/6| < \varepsilon$.


In [ ]:
import numpy as np

target = np.pi**2 / 6
n_max = 2_000_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
partial = np.cumsum(1.0 / ks**2)

for n in [1, 5, 10, 100, 1_000, 10_000, 100_000]:
    print(f'S_{n:>7d} = {partial[n-1]:.10f}   gap = {abs(partial[n-1] - target):.2e}')

gap = np.abs(partial - target)
for eps in [1e-2, 1e-4, 1e-6]:
    idx = np.argmax(gap < eps)
    # Verify monotone: from idx onward, gap stays < eps (true here since gap ~ 1/n).
    assert gap[idx] < eps
    print(f'eps = {eps:.0e}   smallest N = {idx + 1}')


## Cauchy criterion

A real sequence is Cauchy iff for every $\varepsilon > 0$ there exists $N$ with $|a_m - a_n| < \varepsilon$ for all $m, n \geq N$. We test $a_n = 1/n$ (Cauchy) against $b_n = (-1)^n$ (not Cauchy).


In [ ]:
import numpy as np

def is_cauchy(seq, eps):
    """Return smallest N such that sup_{m,n>=N} |seq[m]-seq[n]| < eps, or None."""
    seq = np.asarray(seq, dtype=np.float64)
    L = len(seq)
    for N in range(L):
        tail = seq[N:]
        if tail.max() - tail.min() < eps:
            return N
    return None

a = 1.0 / np.arange(1, 5001)
b = (-1.0) ** np.arange(5000)

for eps in [1e-1, 1e-2, 1e-3]:
    Na = is_cauchy(a, eps)
    Nb = is_cauchy(b, eps)
    print(f'eps = {eps:.0e}   a_n=1/n: N = {Na}    b_n=(-1)^n: N = {Nb}')


## $\sqrt{2} \notin \mathbb{Q}$ and Newton's method

**Proof recap.** If $\sqrt{2} = p/r$ in lowest terms, then $p^2 = 2r^2$ forces $p$ even, then $r$ even, contradicting $\gcd(p,r)=1$.

So $\sqrt{2}$ lives in $\mathbb{R} \setminus \mathbb{Q}$ — and the Newton iteration $x_{n+1} = (x_n + 2/x_n)/2$ produces a Cauchy sequence of rationals whose limit is $\sqrt{2}$, witnessing why we needed completeness in the first place.


In [ ]:
import numpy as np

x = 1.0
true = np.sqrt(2.0)
print(f'{ "n":>3}  {"x_n":>20}  {"|x_n - sqrt(2)|":>18}')
for n in range(11):
    print(f'{n:>3}  {x:>20.16f}  {abs(x - true):>18.2e}')
    x = 0.5 * (x + 2.0 / x)


## Bounded monotone convergence: telescoping series

$a_n = \sum_{k=1}^n \tfrac{1}{k(k+1)} = 1 - \tfrac{1}{n+1}$ is monotone increasing and bounded above by $1$. By Theorem 2.8 it must converge — and indeed to $\sup_n a_n = 1$.


In [ ]:
import numpy as np

n_max = 100_000
ks = np.arange(1, n_max + 1, dtype=np.float64)
terms = 1.0 / (ks * (ks + 1.0))
a = np.cumsum(terms)

monotone = bool(np.all(np.diff(a) >= 0))
bounded = bool(np.all(a <= 1.0))
print(f'monotone increasing: {monotone}')
print(f'bounded above by 1 : {bounded}')
for n in [1, 10, 100, 1_000, 10_000, 100_000]:
    print(f'a_{n:>6d} = {a[n-1]:.10f}   gap to 1 = {1 - a[n-1]:.2e}')
print(f'sup_n a_n (numerical) = {a.max():.12f}')


## Forward link

When we prove SGD converges in Chapter 13, the core lemma is: the loss sequence $(L(\theta_t))$ is monotone decreasing in expectation and bounded below by $0$, hence convergent — by **exactly** Theorem 2.8 of this chapter.


## Continuity, differentiation, chain rule

We numerically explore the four anchors of Chapter 3:
1. $\varepsilon$-$\delta$ continuity at a point.
2. Derivative as a limit (forward difference).
3. Chain rule.
4. Mean value theorem.

**Recall:** $f$ is continuous at $a$ iff $\forall\,\varepsilon>0\;\exists\,\delta>0:\;|x-a|<\delta \Rightarrow |f(x)-f(a)|<\varepsilon$.


In [ ]:
import numpy as np
np.random.seed(0)

# Numerical eps-delta certificate for f(x) = x^2 at a = 2.
# Strategy: for each eps, binary-search the largest delta in (0, 1] for which
# sup_{|x-a|<delta} |f(x)-f(a)| < eps holds (sampled densely).

def f(x):
    return x * x

a = 2.0
fa = f(a)

def violation(delta, n=4001):
    xs = np.linspace(a - delta, a + delta, n)
    return np.max(np.abs(f(xs) - fa))

def find_delta(eps, lo=0.0, hi=1.0, iters=60):
    # halving search: largest hi with violation(hi) < eps
    if violation(hi) < eps:
        return hi
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if violation(mid) < eps:
            lo = mid
        else:
            hi = mid
    return lo

print(f'{"eps":>10} {"delta found":>15} {"sup|f(x)-f(a)|":>20}  ok?')
for eps in (1e-1, 1e-2, 1e-3):
    d = find_delta(eps)
    sup = violation(d)
    print(f'{eps:>10.0e} {d:>15.8e} {sup:>20.8e}  {sup < eps}')


## Derivative as a limit

$f'(a) = \lim_{h\to 0} \frac{f(a+h)-f(a)}{h}$.

The forward-difference truncation error for smooth $f$ is $O(h)$ (Taylor expansion). Below we plot it on a log-log axis if matplotlib is available; otherwise we print a table.


In [ ]:
import numpy as np

# Forward-difference derivative of sin at a grid of points.
xs = np.linspace(0.1, np.pi - 0.1, 200)
true = np.cos(xs)

hs = np.array([10.0 ** k for k in range(-1, -13, -1)])
errors = np.array([np.max(np.abs((np.sin(xs + h) - np.sin(xs)) / h - true)) for h in hs])

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.loglog(hs, errors, 'o-', label='forward diff error')
    ax.loglog(hs, hs, 'k--', alpha=0.5, label='O(h) reference')
    ax.set_xlabel('h'); ax.set_ylabel('max error'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout(); fig.savefig('fd_error.png', dpi=120)
    print('Saved fd_error.png')
except Exception as e:
    print(f'(matplotlib unavailable: {e}) -- printing table instead')

print(f'{"h":>12} {"max |fd - cos|":>20}')
for h, err in zip(hs, errors):
    print(f'{h:>12.1e} {err:>20.6e}')


## Chain rule

If $g$ is differentiable at $a$ and $f$ is differentiable at $g(a)$, then
$$ (f\circ g)'(a) = f'(g(a)) \cdot g'(a). $$

**Sketch (Carath\'eodory):** define $\phi(y)=(f(y)-f(b))/(y-b)$ for $y\ne b$ and $\phi(b)=f'(b)$. Then $\phi$ is continuous at $b=g(a)$ and $f(y)-f(b)=\phi(y)(y-b)$ identically. Substitute $y=g(x)$, divide by $x-a$, take $x\to a$.

We verify on $f(x)=\sin(x^2)$ at $x=1$: analytic value is $\cos(1)\cdot 2 \approx 1.0806$.


In [ ]:
import numpy as np

x = 1.0
F = lambda t: np.sin(t * t)

analytic = np.cos(x * x) * 2.0 * x  # f'(g(x)) * g'(x) with f=sin, g=x^2

# Central difference is O(h^2) and minimizes truncation+roundoff at h ~ 1e-5.
h = 1e-5
numeric = (F(x + h) - F(x - h)) / (2 * h)

print(f'analytic   f\'(g(x)) * g\'(x) = {analytic:.12f}')
print(f'numeric    central diff h=1e-5 = {numeric:.12f}')
print(f'abs error                       = {abs(analytic - numeric):.3e}')
assert abs(analytic - numeric) < 1e-7, 'chain rule check failed'
print('chain rule verified to <1e-7')


## Mean value theorem

If $f$ is continuous on $[a,b]$ and differentiable on $(a,b)$, then $\exists c \in (a,b)$ with
$$ f'(c) = \frac{f(b)-f(a)}{b-a}. $$

We find such a $c$ for $f(x)=x^3-2x$ on $[0,2]$ by bisection on $f'(x)-\text{slope}$.


In [ ]:
import numpy as np

def f(x):  return x**3 - 2*x
def fp(x): return 3*x**2 - 2

a, b = 0.0, 2.0
slope = (f(b) - f(a)) / (b - a)
g = lambda x: fp(x) - slope

# fp is continuous; g(0) = -2 - 1 = -3, g(2) = 10 - 1 = 9, sign change => bisection works.
lo, hi = a, b
assert g(lo) * g(hi) < 0
for _ in range(80):
    mid = 0.5 * (lo + hi)
    if g(lo) * g(mid) <= 0:
        hi = mid
    else:
        lo = mid
c = 0.5 * (lo + hi)

print(f'slope (f(b)-f(a))/(b-a) = {slope:.10f}')
print(f'witness c              = {c:.10f}')
print(f'f\'(c)                  = {fp(c):.10f}')
print(f'|f\'(c) - slope|        = {abs(fp(c) - slope):.3e}')
# Closed form: 3c^2 - 2 = 2  =>  c = sqrt(4/3)
print(f'closed form sqrt(4/3)   = {np.sqrt(4/3):.10f}')


## Multivariate calculus: partials, gradients, Jacobians

We move from $f:\mathbb{R}\to\mathbb{R}$ (Chapter 3) to $f:\mathbb{R}^n\to\mathbb{R}^m$. The right notion of differentiability is **Fréchet**: there exists a linear $L$ with $\|f(\mathbf{a}+\mathbf{h})-f(\mathbf{a})-L\mathbf{h}\| = o(\|\mathbf{h}\|)$. The matrix of $L$ is the **Jacobian** $J_f(\mathbf{a}) \in \mathbb{R}^{m\times n}$, whose $j$-th column is $\partial f/\partial x_j(\mathbf{a})$ (Theorem 4.5). When $m=1$, $\nabla f = J_f^\top$.

Below we verify the key facts numerically with finite differences.

In [ ]:
import numpy as np
np.random.seed(0)

# f: R^2 -> R, f(x,y) = x^2 y + sin(x+y)
def f(x, y):
    return x**2 * y + np.sin(x + y)

# Analytic gradient: df/dx = 2xy + cos(x+y), df/dy = x^2 + cos(x+y)
def grad_f_analytic(x, y):
    return np.array([2*x*y + np.cos(x+y), x**2 + np.cos(x+y)])

def grad_f_numeric(x, y, h=1e-6):
    dx = (f(x+h, y) - f(x-h, y)) / (2*h)
    dy = (f(x, y+h) - f(x, y-h)) / (2*h)
    return np.array([dx, dy])

x0, y0 = 1.0, 2.0
ga = grad_f_analytic(x0, y0)
gn = grad_f_numeric(x0, y0)
print(f'analytic grad at (1,2): {ga}')
print(f'numeric  grad at (1,2): {gn}')
print(f'max abs error: {np.max(np.abs(ga-gn)):.2e}')

## Jacobians by column

For $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^m$, the $j$-th column of $J_f(\mathbf{a})$ is $\partial \mathbf{f}/\partial x_j(\mathbf{a})$, computable as the central difference $(\mathbf{f}(\mathbf{a}+h\mathbf{e}_j)-\mathbf{f}(\mathbf{a}-h\mathbf{e}_j))/(2h)$. We test on $\mathbf{f}(x,y,z)=(xyz,\ x^2+\sin y+e^z)$, whose analytic Jacobian is

$$J_f = \begin{pmatrix} yz & xz & xy \\ 2x & \cos y & e^z \end{pmatrix}.$$

In [ ]:
import numpy as np

def F(v):
    x, y, z = v
    return np.array([x*y*z, x**2 + np.sin(y) + np.exp(z)])

def J_analytic(v):
    x, y, z = v
    return np.array([
        [y*z,      x*z,      x*y],
        [2*x,      np.cos(y), np.exp(z)],
    ])

def J_numeric(F, v, h=1e-6):
    v = np.asarray(v, dtype=float)
    n = v.size
    cols = []
    for j in range(n):
        ej = np.zeros(n); ej[j] = 1.0
        cols.append((F(v + h*ej) - F(v - h*ej)) / (2*h))
    return np.stack(cols, axis=1)

v0 = np.array([1.0, 0.5, -0.3])
Ja = J_analytic(v0)
Jn = J_numeric(F, v0)
print('analytic J:'); print(Ja)
print('numeric  J:'); print(Jn)
print(f'max abs error: {np.max(np.abs(Ja-Jn)):.2e}')

## Multivariate chain rule

Theorem 4.7: $J_{f\circ g}(\mathbf{a}) = J_f(g(\mathbf{a}))\, J_g(\mathbf{a})$. Take $g(t)=(\cos t,\sin t)$ and $f(x,y)=x^2+y^2$. Then $f\circ g \equiv 1$, so $(f\circ g)'(t)=0$ for every $t$. The chain rule must produce the same answer.

In [ ]:
import numpy as np

def g(t):
    return np.array([np.cos(t), np.sin(t)])

def gprime(t):
    return np.array([-np.sin(t), np.cos(t)])  # column vector (R -> R^2 has 2x1 Jacobian)

def grad_f(p):
    x, y = p
    return np.array([2*x, 2*y])  # row of J_f

ts = np.linspace(0, 2*np.pi, 7)
for t in ts:
    direct = 0.0  # f(g(t)) = 1, derivative is 0
    chain  = float(grad_f(g(t)) @ gprime(t))   # 1x2 @ 2x1
    print(f't={t:6.3f}  direct={direct:+.2e}  chain-rule={chain:+.2e}')

## Schwarz / Clairaut: equality of mixed partials

For $C^2$ functions, $\partial_x\partial_y f = \partial_y\partial_x f$ (Theorem 4.8). Numerically, both can be approximated by the second-order central difference

$$\partial_x\partial_y f(a,b) \approx \frac{f(a+h,b+k)-f(a+h,b-k)-f(a-h,b+k)+f(a-h,b-k)}{4hk}.$$

We test on $f(x,y) = x^3 y^2 + \sin(xy)$ at $(1,1)$. Analytically,
$\partial_y f = 2x^3 y + x\cos(xy)$, so $\partial_x\partial_y f = 6x^2 y + \cos(xy) - xy\sin(xy)$, which at $(1,1)$ is $6 + \cos(1) - \sin(1)$.

In [ ]:
import numpy as np

def fxy(x, y):
    return x**3 * y**2 + np.sin(x*y)

def mixed_partial(F, a, b, h=1e-3, k=1e-3):
    return (F(a+h, b+k) - F(a+h, b-k) - F(a-h, b+k) + F(a-h, b-k)) / (4*h*k)

a, b = 1.0, 1.0
dxdy = mixed_partial(fxy, a, b)
dydx = mixed_partial(lambda y, x: fxy(x, y), b, a)  # swap roles
analytic = 6*a**2*b + np.cos(a*b) - a*b*np.sin(a*b)

print(f'numeric  d/dx d/dy f at (1,1): {dxdy:.10f}')
print(f'numeric  d/dy d/dx f at (1,1): {dydx:.10f}')
print(f'analytic value             : {analytic:.10f}')
print(f'|dxdy - dydx| = {abs(dxdy-dydx):.2e}')
print(f'|num - analytic| = {abs(dxdy-analytic):.2e}')

## Connection to LLMs

A transformer is a composition $F = F_L\circ\cdots\circ F_1$. Theorem 4.7 says the gradient of the loss with respect to layer-$\ell$ parameters is a product of Jacobians from the loss back to that layer. Backpropagation never materializes those matrices: it propagates a *row vector* $\mathbf{v}^\top$ right-to-left via vector–Jacobian products (VJPs), one per layer, each at $O(\text{forward cost})$. We will derive reverse-mode autodiff formally in Chapter 18.

# Chapter 5 — Linear algebra I: vector spaces, basis, linear maps

Every transformer layer is a linear map between finite-dimensional real vector spaces, framed by bias terms and nonlinearities. Before attention (Ch. 21) or embeddings (Ch. 19), we need vector spaces, bases, dimension, kernels, images, and the rank–nullity theorem.

**Eight axioms of a vector space $V$ over a field $\mathbb{F}$.** For $u, v, w \in V$, $a, b \in \mathbb{F}$:

1. $(u+v)+w = u+(v+w)$
2. $u+v = v+u$
3. $\exists\, 0 \in V$ with $v+0=v$
4. $\forall v\,\exists (-v)$ with $v+(-v)=0$
5. $a(u+v) = au + av$
6. $(a+b)v = av + bv$
7. $(ab)v = a(bv)$
8. $1\cdot v = v$

The canonical example is $V = \mathbb{R}^d$ over $\mathbb{F} = \mathbb{R}$, which is exactly the embedding space used by language models with hidden dimension $d$.

In [ ]:
import numpy as np

def is_linearly_independent(vectors):
    """Vectors is a list/array of row vectors. Independent iff rank == count."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M)) == M.shape[0]

def dim_span(vectors):
    """Dimension of the span of a list of vectors."""
    M = np.asarray(vectors, dtype=float)
    return int(np.linalg.matrix_rank(M))

S = [(1, 0, 0), (0, 1, 0), (1, 1, 0)]
print('vectors:', S)
print('linearly independent?', is_linearly_independent(S))
print('dim of span:', dim_span(S))
assert dim_span(S) == 2, 'expected 2 since (1,1,0) = (1,0,0) + (0,1,0)'

## Basis and dimension

A **basis** of $V$ is a linearly independent spanning set. By the **Steinitz exchange lemma**, every basis of a finite-dimensional $V$ has the same cardinality, called $\dim V$. Below we compute the rank of a $4 \times 6$ matrix (the dimension of its column span / image) and extract a basis of its **null space** (kernel) via the right singular vectors of $A$ associated to zero singular values.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
print('A =\n', A)

U, sigma, Vt = np.linalg.svd(A)
print('singular values:', np.round(sigma, 4))

# Numerical rank: count singular values above tolerance.
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
print('rank(A) =', rank)

# Right singular vectors (rows of Vt) corresponding to zero singular values
# span the null space. There are n - rank of them, where n = 6.
n = A.shape[1]
null_basis = Vt[rank:]  # shape (n - rank, n)
print('dim ker(A) =', null_basis.shape[0])

# Verify A @ v == 0 for each null-space basis vector v.
for i, v in enumerate(null_basis):
    Av = A @ v
    print(f'A v_{i} norm = {np.linalg.norm(Av):.2e}')
    assert np.allclose(Av, 0, atol=1e-10)

## Linear maps and matrix representation

A linear map $T: V \to W$ satisfies $T(au+bv) = aTu + bTv$. In bases $(e_j)$ of $V = \mathbb{R}^n$ and $(f_i)$ of $W = \mathbb{R}^m$, the matrix $A \in \mathbb{R}^{m\times n}$ has $j$-th column equal to the coordinates of $T(e_j)$. If we change basis on $V = W = \mathbb{R}^n$ via an invertible $P$, the same map $T$ is represented in the new basis by $\tilde{A} = P^{-1} A P$. Numerically: for any $v \in \mathbb{R}^n$ we should have $A v = P\,\tilde{A}\,P^{-1} v$.

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randn(3, 3)
# Build a random invertible P. Re-draw if singular (essentially never happens).
while True:
    P = np.random.randn(3, 3)
    if abs(np.linalg.det(P)) > 1e-6:
        break
P_inv = np.linalg.inv(P)
A_tilde = P_inv @ A @ P

v = np.random.randn(3)
lhs = A @ v
rhs = P @ A_tilde @ P_inv @ v
print('A v       =', lhs)
print('P A~ Pi v =', rhs)
print('max abs diff:', np.max(np.abs(lhs - rhs)))
assert np.allclose(lhs, rhs)

## Rank–nullity

**Theorem.** For $T: V \to W$ with $\dim V < \infty$, $\dim \ker T + \dim \mathrm{im}\,T = \dim V$. Equivalently, for $A \in \mathbb{R}^{m\times n}$, $\mathrm{rank}(A) + \dim \ker(A) = n$. We verify this for the $4 \times 6$ matrix above (so the sum should equal $6$).

In [ ]:
import numpy as np
np.random.seed(0)

A = np.random.randint(-3, 4, size=(4, 6)).astype(float)
U, sigma, Vt = np.linalg.svd(A)
tol = max(A.shape) * np.finfo(float).eps * sigma.max()
rank = int((sigma > tol).sum())
nullity = A.shape[1] - rank  # by definition of SVD null-space basis
print(f'rank(A)    = {rank}')
print(f'nullity(A) = {nullity}')
print(f'sum        = {rank + nullity}')
print(f'n (cols)   = {A.shape[1]}')
assert rank + nullity == A.shape[1], 'rank-nullity violated'

## Connection to LLMs

A transformer with hidden dimension $d$ operates in $\mathbb{R}^d$. The query/key/value projections in attention (Ch. 21) are linear maps $\mathbb{R}^d \to \mathbb{R}^{d_k}$. Token embeddings (Ch. 19) are a linear map $\mathbb{R}^{|\mathcal{V}|} \to \mathbb{R}^d$ applied to a one-hot input. LoRA constrains weight *updates* to a low-rank subspace, an explicit application of $\dim \mathrm{im}\,T \le \min(m, n)$. Rank–nullity will reappear whenever we count free parameters or degrees of freedom.

## Inner products, norms, and Cauchy-Schwarz

The dot product on $\mathbb{R}^n$ is $\langle x,y\rangle=\sum_i x_iy_i$, and its induced norm is $\|x\|=\sqrt{\langle x,x\rangle}$. Cauchy-Schwarz says $|\langle x,y\rangle|\le\|x\|\,\|y\|$. We numerically verify this on 50 random pairs in $\mathbb{R}^{10}$ by computing the ratio $|\langle x,y\rangle|/(\|x\|\|y\|)$ -- it must always be $\le 1$.


In [ ]:
import numpy as np

np.random.seed(0)
n_pairs, dim = 50, 10
ratios = []
for _ in range(n_pairs):
    x = np.random.randn(dim)
    y = np.random.randn(dim)
    inner = float(np.dot(x, y))
    nx = float(np.linalg.norm(x))
    ny = float(np.linalg.norm(y))
    ratios.append(abs(inner) / (nx * ny))

ratios = np.array(ratios)
print(f'pairs tested        : {n_pairs}')
print(f'max  |<x,y>|/(|x||y|): {ratios.max():.6f}')
print(f'mean |<x,y>|/(|x||y|): {ratios.mean():.6f}')
assert ratios.max() <= 1.0 + 1e-12, 'Cauchy-Schwarz violated!'
print('Cauchy-Schwarz holds for all 50 pairs.')


## Eigenvalues of symmetric matrices

The spectral theorem says every real symmetric matrix $A$ has an orthonormal eigenbasis: $A=Q\Lambda Q^\top$. Below we build a symmetric $5\times5$ matrix $A=M+M^\top$, diagonalize with `np.linalg.eigh` (which exploits symmetry), and verify both $A v_i=\lambda_i v_i$ for every $i$ and $V^\top V=I$ (orthonormal eigenvectors).


In [ ]:
import numpy as np

np.random.seed(0)
M = np.random.randn(5, 5)
A = M + M.T
assert np.allclose(A, A.T)

eigvals, V = np.linalg.eigh(A)
print('eigenvalues:', np.round(eigvals, 6))

max_eig_residual = 0.0
for i in range(A.shape[0]):
    lhs = A @ V[:, i]
    rhs = eigvals[i] * V[:, i]
    max_eig_residual = max(max_eig_residual, float(np.linalg.norm(lhs - rhs)))
print(f'max ||A v_i - lambda_i v_i|| : {max_eig_residual:.2e}')

ortho_err = float(np.linalg.norm(V.T @ V - np.eye(5)))
print(f'||V^T V - I||_F              : {ortho_err:.2e}')

assert max_eig_residual < 1e-10
assert ortho_err < 1e-10
print('Spectral theorem verified numerically.')


## Singular value decomposition

Every $A\in\mathbb{R}^{m\times n}$ admits an SVD $A=U\Sigma V^\top$ with $U,V$ orthogonal and $\Sigma$ diagonal with nonnegative entries. We build a random $5\times3$ matrix, run `np.linalg.svd`, and reconstruct $A$ to high precision.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

Sigma = np.zeros_like(A)
Sigma[:len(s), :len(s)] = np.diag(s)
A_reco = U @ Sigma @ Vt

print('singular values        :', np.round(s, 6))
print('U shape, Sigma shape, Vt shape:', U.shape, Sigma.shape, Vt.shape)
print(f'||A - U Sigma V^T||_F  : {np.linalg.norm(A - A_reco):.2e}')
print(f'||U^T U - I||_F        : {np.linalg.norm(U.T @ U - np.eye(5)):.2e}')
print(f'||V V^T - I||_F        : {np.linalg.norm(Vt @ Vt.T - np.eye(3)):.2e}')

assert np.linalg.norm(A - A_reco) < 1e-10
print('SVD reconstruction verified.')


## Eckart-Young: best low-rank approximation

Eckart-Young says the best rank-$k$ approximation of $A$ in Frobenius norm is the truncated SVD $A_k=\sum_{i=1}^k\sigma_i u_i v_i^\top$, with squared error $\sum_{i>k}\sigma_i^2$. We compute rank-1 and rank-2 truncations of the same $5\times3$ matrix and observe a strict monotone decrease in error, matching the predicted tail sum of squared singular values.


In [ ]:
import numpy as np

np.random.seed(0)
A = np.random.randn(5, 3)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

errors = {}
for k in (1, 2, 3):
    A_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    err = float(np.linalg.norm(A - A_k))
    predicted = float(np.sqrt(np.sum(s[k:] ** 2)))
    errors[k] = err
    print(f'rank {k}: ||A - A_k||_F = {err:.6f}   predicted sqrt(sum sigma_i^2 for i>k) = {predicted:.6f}')

assert errors[1] >= errors[2] >= errors[3]
assert errors[3] < 1e-10
print('Frobenius error decreases monotonically with rank, as Eckart-Young predicts.')


# Chapter 7 — Convexity and gradient descent

Why convex? Because it is the **only** setting where we can write down honest convergence rates for $x_{t+1} = x_t - \eta \nabla f(x_t)$. Real transformer losses are non-convex (Chapter 13, 14, 27), but the convex rates supply the vocabulary we use to reason about them: $L$-smoothness, condition number $\kappa = L/\mu$, contraction.

**Definitions.** $f$ is *convex* iff $f(t x + (1-t) y) \le t f(x) + (1-t) f(y)$. It is *$L$-smooth* iff $\|\nabla f(x) - \nabla f(y)\| \le L\|x - y\|$. It is *$\mu$-strongly convex* iff $f(y) \ge f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{\mu}{2}\|y - x\|^2$.


In [ ]:
import numpy as np

# A strongly convex quadratic f(x, y) = (x - 1)^2 + 2 (y + 1)^2
# Hessian = diag(2, 4), so mu = 2, L = 4, kappa = 2, minimum at x* = (1, -1).
def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2

def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

x_star = np.array([1.0, -1.0])

# Numerical convexity check: midpoint inequality on random pairs.
np.random.seed(0)
violations = 0
for _ in range(2000):
    a, b = np.random.randn(2) * 3, np.random.randn(2) * 3
    t = np.random.rand()
    if f(t * a + (1 - t) * b) > t * f(a) + (1 - t) * f(b) + 1e-9:
        violations += 1
print(f'Convexity violations in 2000 samples: {violations}')

# Contour plot, with table fallback if matplotlib is unavailable.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    xs = np.linspace(-2, 4, 80); ys = np.linspace(-4, 2, 80)
    X, Y = np.meshgrid(xs, ys)
    Z = (X - 1.0)**2 + 2.0 * (Y + 1.0)**2
    plt.figure(figsize=(5, 4))
    plt.contour(X, Y, Z, levels=20)
    plt.scatter([1], [-1], c='red', label='x*')
    plt.title('f(x,y) = (x-1)^2 + 2(y+1)^2'); plt.legend()
    plt.savefig('ch07_contour.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved contour to ch07_contour.png')
except Exception as e:
    print(f'matplotlib unavailable ({e}); printing slice table:')
    for x in [-1, 0, 1, 2, 3]:
        row = [f(np.array([x, y])) for y in [-3, -2, -1, 0, 1]]
        print(f'x={x:+d}: ' + '  '.join(f'{v:6.2f}' for v in row))


## Descent lemma and the $1/L$ step

**Descent lemma.** $L$-smoothness implies $f(y) \le f(x) + \langle \nabla f(x), y - x\rangle + \tfrac{L}{2}\|y - x\|^2$. Plugging $y = x - \tfrac{1}{L}\nabla f(x)$ yields
$$f(x_{t+1}) \le f(x_t) - \tfrac{1}{2L}\|\nabla f(x_t)\|^2,$$
i.e. **monotone descent**. For $\mu$-strongly convex $f$ this strengthens to $\|x_{t+1} - x^*\|^2 \le (1 - \mu/L)\|x_t - x^*\|^2$, a *geometric* contraction.


In [ ]:
import numpy as np

def f(z):
    return (z[0] - 1.0)**2 + 2.0 * (z[1] + 1.0)**2
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])
x_star = np.array([1.0, -1.0])

# GD on the strongly convex quadratic with eta = 1/L.
L = 4.0; mu = 2.0; eta = 1.0 / L
x = np.array([3.5, 1.5])
history = []
for t in range(200):
    history.append(np.linalg.norm(x - x_star)**2)
    x = x - eta * grad_f(x)
history.append(np.linalg.norm(x - x_star)**2)

predicted_factor = 1.0 - mu / L  # 0.5
print(f'Predicted contraction per step: {predicted_factor}')
for t in [0, 1, 2, 5, 10, 20, 50, 100, 200]:
    pred = history[0] * predicted_factor**t
    print(f't={t:3d}  ||x_t - x*||^2 = {history[t]:.3e}   predicted bound = {pred:.3e}')

ratios = [history[t+1] / history[t] for t in range(50) if history[t] > 1e-30]
print(f'Median empirical per-step ratio over first 50 steps: {np.median(ratios):.4f}')


## $O(1/T)$ rate without strong convexity

If $f$ is $L$-smooth and convex but **not** $\mu$-strongly convex, the rate degrades from geometric to $f(x_T) - f^* \le \tfrac{L \|x_0 - x^*\|^2}{2T}$. We exhibit this on a least-squares problem $f(x) = \|Ax - b\|^2$ where $A \in \mathbb{R}^{20 \times 10}$ has a tiny smallest singular value: in the slow direction the effective $\mu$ is essentially zero.


In [ ]:
import numpy as np

np.random.seed(0)
U, _ = np.linalg.qr(np.random.randn(20, 20))
V, _ = np.linalg.qr(np.random.randn(10, 10))
sigma = np.linspace(1.0, 1e-3, 10)  # smallest singular value 1e-3 -> tiny mu
S = np.zeros((20, 10)); np.fill_diagonal(S, sigma)
A = U @ S @ V.T
b = np.random.randn(20)

x_star_ls, *_ = np.linalg.lstsq(A, b, rcond=None)
f_star = float(np.linalg.norm(A @ x_star_ls - b)**2)

L_ls = 2.0 * (sigma.max()**2)
print(f'sigma_max={sigma.max():.4f}, sigma_min={sigma.min():.4f}, L={L_ls:.4f}')

x = np.zeros(10); eta = 1.0 / L_ls
gaps = []
for t in range(1000):
    r = A @ x - b
    g = 2.0 * (A.T @ r)
    x = x - eta * g
    gaps.append(float(np.linalg.norm(A @ x - b)**2) - f_star)

Ts = [1, 2, 5, 10, 50, 100, 500, 1000]
print('   T       f(x_T)-f*        bound L||x0-x*||^2/(2T)')
C = L_ls * float(np.linalg.norm(x_star_ls)**2) / 2.0
for T in Ts:
    print(f'  {T:4d}    {gaps[T-1]:.3e}      {C/T:.3e}')

tail_T = np.arange(100, 1001)
tail_g = np.array(gaps[99:1000])
tail_g = np.maximum(tail_g, 1e-20)
slope, intercept = np.polyfit(np.log(tail_T), np.log(tail_g), 1)
print(f'log-log slope on T in [100, 1000]: {slope:.3f}  (theory predicts ~ -1)')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.loglog(np.arange(1, 1001), np.maximum(gaps, 1e-20), label='f(x_T) - f*')
    plt.loglog(np.arange(1, 1001), C / np.arange(1, 1001), '--', label='L||x0-x*||^2/(2T)')
    plt.xlabel('T'); plt.ylabel('suboptimality'); plt.legend(); plt.title('GD on ill-conditioned LS')
    plt.savefig('ch07_rate.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved log-log rate plot to ch07_rate.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Strong convexity: $\eta = 1/L$ vs. the optimal $\eta = 2/(L + \mu)$

For a quadratic with eigenvalues in $[\mu, L]$ the per-step contraction with $\eta = 1/L$ is $1 - \mu/L$, while with the optimal $\eta = 2/(L + \mu)$ it improves to $((\kappa - 1)/(\kappa + 1))^2$ where $\kappa = L/\mu$. Both rates are linear; the optimal one has a strictly smaller constant.


In [ ]:
import numpy as np
L = 4.0; mu = 2.0
x_star = np.array([1.0, -1.0])
def grad_f(z):
    return np.array([2.0 * (z[0] - 1.0), 4.0 * (z[1] + 1.0)])

def run(eta, T=40):
    x = np.array([3.5, 1.5])
    err = [np.linalg.norm(x - x_star)**2]
    for _ in range(T):
        x = x - eta * grad_f(x)
        err.append(np.linalg.norm(x - x_star)**2)
    return err

eta_basic = 1.0 / L
eta_opt = 2.0 / (L + mu)
h_basic = run(eta_basic)
h_opt = run(eta_opt)
kappa = L / mu
rate_basic = 1.0 - mu / L
rate_opt = ((kappa - 1.0) / (kappa + 1.0))**2
print(f'theory: eta=1/L contracts by {rate_basic:.3f}/step; eta=2/(L+mu) by {rate_opt:.3f}/step')
print(f'  t   ||x_t-x*||^2 (1/L)    ||x_t-x*||^2 (2/(L+mu))')
for t in [0, 5, 10, 20, 40]:
    print(f'  {t:3d}     {h_basic[t]:.3e}             {h_opt[t]:.3e}')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    plt.semilogy(h_basic, label='eta = 1/L')
    plt.semilogy(h_opt, label='eta = 2/(L+mu) (optimal)')
    plt.xlabel('t'); plt.ylabel('||x_t - x*||^2'); plt.legend(); plt.title('Step-size comparison')
    plt.savefig('ch07_steps.png', dpi=80, bbox_inches='tight'); plt.close()
    print('Saved step-size comparison to ch07_steps.png')
except Exception as e:
    print(f'(plot skipped: {e})')


## Forward to the LLM chapters

Transformer training is non-convex, but the same skeleton recurs: a descent-lemma-style inequality (now in expectation) plus a contraction or telescoping argument. SGD (Chapter 13) replaces $(\star)$ with $\mathbb{E}[f(x_{t+1})] \le f(x_t) - \tfrac{\eta}{2}\|\nabla f(x_t)\|^2 + \tfrac{L \eta^2 \sigma^2}{2}$. AdamW (Chapter 14) builds preconditioners on top. The pre-training pipeline (Chapter 27) uses warmup precisely because the *local* $L$ is large early in training, exactly the regime where the descent lemma forbids a large step.


# Block B — Probability and Information


<!-- CHAPTER 8 START -->
## Chapter 8: Probability foundations: sample spaces, sigma-algebras, Kolmogorov axioms

_Stub — will be filled by Phase 2._
<!-- CHAPTER 8 END -->

<!-- CHAPTER 9 START -->
## Chapter 9: Random variables, distributions, CDF/PMF/PDF

_Stub — will be filled by Phase 2._
<!-- CHAPTER 9 END -->

<!-- CHAPTER 10 START -->
## Chapter 10: Expectation, variance, covariance; Jensen's inequality

_Stub — will be filled by Phase 2._
<!-- CHAPTER 10 END -->

<!-- CHAPTER 11 START -->
## Chapter 11: Information theory: self-information, entropy, cross-entropy, KL

_Stub — will be filled by Phase 2._
<!-- CHAPTER 11 END -->

<!-- CHAPTER 12 START -->
## Chapter 12: Statistical inference: likelihood, MLE, ERM, bias-variance

_Stub — will be filled by Phase 2._
<!-- CHAPTER 12 END -->

# Block C — Stochastic Optimization


<!-- CHAPTER 13 START -->
## Chapter 13: SGD: stochastic-approximation theorem; mini-batching; convergence sketch

_Stub — will be filled by Phase 2._
<!-- CHAPTER 13 END -->

<!-- CHAPTER 14 START -->
## Chapter 14: Momentum, RMSProp, AdamW: derivation and bias-correction proof

_Stub — will be filled by Phase 2._
<!-- CHAPTER 14 END -->

# Block D — Neural Networks


<!-- CHAPTER 15 START -->
## Chapter 15: MLPs as compositional functions; universal approximation

_Stub — will be filled by Phase 2._
<!-- CHAPTER 15 END -->

<!-- CHAPTER 16 START -->
## Chapter 16: Activation functions: ReLU/GELU/softmax with derivatives

_Stub — will be filled by Phase 2._
<!-- CHAPTER 16 END -->

<!-- CHAPTER 17 START -->
## Chapter 17: Loss functions: MSE, cross-entropy; gradients from first principles

_Stub — will be filled by Phase 2._
<!-- CHAPTER 17 END -->

<!-- CHAPTER 18 START -->
## Chapter 18: Backpropagation: chain rule applied; reverse-mode AD as a graph algorithm

_Stub — will be filled by Phase 2._
<!-- CHAPTER 18 END -->

# Block E — Sequence Models and Attention


<!-- CHAPTER 19 START -->
## Chapter 19: Embeddings: token to vector; lookup as a linear map; weight tying

_Stub — will be filled by Phase 2._
<!-- CHAPTER 19 END -->

<!-- CHAPTER 20 START -->
## Chapter 20: RNN intuition; vanishing-gradient proof; why we need attention

_Stub — will be filled by Phase 2._
<!-- CHAPTER 20 END -->

<!-- CHAPTER 21 START -->
## Chapter 21: Scaled dot-product attention: derivation, softmax-temperature analysis

_Stub — will be filled by Phase 2._
<!-- CHAPTER 21 END -->

<!-- CHAPTER 22 START -->
## Chapter 22: Multi-head attention: parallel heads as concat-then-project; complexity

_Stub — will be filled by Phase 2._
<!-- CHAPTER 22 END -->

<!-- CHAPTER 23 START -->
## Chapter 23: Transformer block: residual + LayerNorm/RMSNorm + FFN + attention; gradient-flow argument

_Stub — will be filled by Phase 2._
<!-- CHAPTER 23 END -->

<!-- CHAPTER 24 START -->
## Chapter 24: Positional encoding: sinusoidal derivation, RoPE construction

_Stub — will be filled by Phase 2._
<!-- CHAPTER 24 END -->

# Block F — Pre-training


<!-- CHAPTER 25 START -->
## Chapter 25: Causal masking; next-token prediction loss as MLE on the empirical distribution

_Stub — will be filled by Phase 2._
<!-- CHAPTER 25 END -->

<!-- CHAPTER 26 START -->
## Chapter 26: Tokenization: BPE algorithm; greedy merge correctness

_Stub — will be filled by Phase 2._
<!-- CHAPTER 26 END -->

<!-- CHAPTER 27 START -->
## Chapter 27: Pre-training pipeline: AdamW + warmup + cosine decay + gradient clipping; tiny-GPT training run

_Stub — will be filled by Phase 2._
<!-- CHAPTER 27 END -->

# Block G — Post-training


<!-- CHAPTER 28 START -->
## Chapter 28: SFT, RLHF (PPO/GRPO), and DPO; train + post-train a tiny GPT

_Stub — will be filled by Phase 2._
<!-- CHAPTER 28 END -->